In [1]:
# Metric-agnostic TTD / TTM aggregation pipeline (§1 → §5).
# Public entrypoint: compute_metric_scorecard(scenario_file, metric_name='ttd' | 'ttm' | …)
# Each step stays a standalone function so they can be inspected individually.
import json
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')
SCENARIO_DIR = Path('./data/scenarios')

IMBALANCE_THRESHOLD = 0.15
SKEW_THRESHOLD = 0.15


# ───────── §2: per-observation normalization (piecewise SLA-aware) ─────────
def normalize_score(raw_value: float, sla: float) -> float:
    ratio = raw_value / sla
    if ratio <= 1.0:
        return 1 - 0.85 * ratio
    return max(0.0, 0.15 - 0.3 * (ratio - 1.0))


# ───────── §1: atomic observation (metric-agnostic) ─────────
@dataclass
class Observation:
    run_id: str
    category: str
    sub_fault: str
    metric_name: str
    raw_value: Optional[float]
    sla: Optional[float]
    status: str               # VALID | MISSING | INVALID_ZERO | NO_SLA
    normalized_score: Optional[float]
    sla_compliant: Optional[bool]


def build_observation(run_id, category, sub_fault, raw_value, sla, metric_name='ttd'):
    if sla is None:
        return Observation(run_id, category, sub_fault, metric_name, raw_value, sla,
                           'NO_SLA', None, None)
    if raw_value is None:
        return Observation(run_id, category, sub_fault, metric_name, raw_value, sla,
                           'MISSING', 0.0, False)
    if raw_value <= 0:
        return Observation(run_id, category, sub_fault, metric_name, raw_value, sla,
                           'INVALID_ZERO', 0.0, False)
    return Observation(run_id, category, sub_fault, metric_name, raw_value, sla,
                       'VALID', normalize_score(raw_value, sla), raw_value <= sla)


def load_observations(scenario_file: Path, metric_name: str = 'ttd') -> pd.DataFrame:
    with open(scenario_file) as f:
        scenario = json.load(f)
    sla_map = scenario.get('sla', {})
    obs = [
        build_observation(run_id, category, sub_fault, raw_value,
                          sla_map.get(sub_fault), metric_name=metric_name)
        for run_id, categories in scenario.get('runs', {}).items()
        for category, faults in categories.items()
        for sub_fault, raw_value in faults.items()
    ]
    df = pd.DataFrame([asdict(o) for o in obs])
    df['ratio'] = df.apply(
        lambda r: r['raw_value'] / r['sla'] if r['status'] == 'VALID' else None, axis=1)
    df['run_id'] = df['run_id'].str.slice(0, 8)
    return df[['run_id', 'category', 'sub_fault', 'metric_name', 'raw_value', 'sla',
               'ratio', 'status', 'normalized_score', 'sla_compliant']]


# ───────── §3: sub-fault grain (confidence-tiered) ─────────
def aggregate_subfault(df: pd.DataFrame) -> pd.DataFrame:
    pool = df[df.status != 'NO_SLA']
    rows = []
    for (category, sub_fault), g in pool.groupby(['category', 'sub_fault'], sort=False):
        scores = g['normalized_score'].fillna(0).to_numpy()
        n = len(scores)
        n_valid = int((g['status'] == 'VALID').sum())
        n_compliant = int(g['sla_compliant'].fillna(False).sum())
        if n < 3:
            weighted, confidence = None, 'INSUFFICIENT'
        elif n < 5:
            weighted, confidence = float(scores.mean()), 'LOW'
        elif n < 20:
            weighted = 0.7 * np.median(scores) + 0.3 * np.percentile(scores, 5)
            confidence = 'MEDIUM'
        else:
            weighted = (0.5 * np.median(scores)
                        + 0.3 * np.percentile(scores, 5)
                        + 0.2 * np.percentile(scores, 1))
            confidence = 'HIGH'
        rows.append({
            'category': category,
            'sub_fault': sub_fault,
            'n_attempted': n,
            'detection_rate': round(n_valid / n, 3),
            'sla_compliance': round(n_compliant / n, 3),
            'weighted_score': None if weighted is None else round(float(weighted), 3),
            'median': round(float(np.median(scores)), 3),
            'p5': round(float(np.percentile(scores, 5)), 3),
            'p1': round(float(np.percentile(scores, 1)), 3),
            'confidence': confidence,
        })
    return pd.DataFrame(rows)


# ───────── §4: category grain ─────────
def aggregate_category(df: pd.DataFrame) -> pd.DataFrame:
    pool = df[df.status != 'NO_SLA']
    rows = []
    for category, g in pool.groupby('category', sort=False):
        scores = g['normalized_score'].fillna(0).to_numpy()
        n = len(scores)
        n_valid = int((g['status'] == 'VALID').sum())
        n_compliant = int(g['sla_compliant'].fillna(False).sum())
        rows.append({
            'category': category,
            'n_sub_faults': g['sub_fault'].nunique(),
            'n_attempted': n,
            'detection_rate': round(n_valid / n, 3),
            'sla_compliance': round(n_compliant / n, 3),
            'category_score': round(float(np.median(scores)), 3),
        })
    return pd.DataFrame(rows)


# ───────── §5: cumulative (agent-level) ─────────
def aggregate_cumulative(df: pd.DataFrame, cat_agg: pd.DataFrame) -> pd.DataFrame:
    """Headline-only view. Diagnostic columns (mean, skew/imbalance deltas, n_categories,
    n_no_sla_excluded) collapsed into a single `quality_flags` list."""
    pool = df[df.status != 'NO_SLA']
    scores = pool['normalized_score'].fillna(0).to_numpy()
    n = len(scores)
    n_valid = int((pool['status'] == 'VALID').sum())
    n_compliant = int(pool['sla_compliant'].fillna(False).sum())
    n_no_sla = int((df.status == 'NO_SLA').sum())

    median_score = float(np.median(scores))
    mean_score = float(scores.mean())
    mean_of_cats = float(cat_agg['category_score'].mean())

    flags = []
    if abs(mean_score - median_score) > SKEW_THRESHOLD:
        flags.append('skewed_distribution')
    if abs(mean_of_cats - median_score) > IMBALANCE_THRESHOLD:
        flags.append('mixed_category_health')
    if n_no_sla > 0:
        flags.append(f'{n_no_sla}_obs_excluded_no_sla')

    return pd.DataFrame([{
        'cumulative_score': round(median_score, 3),
        'detection_rate': round(n_valid / n, 3),
        'sla_compliance': round(n_compliant / n, 3),
        'n_attempted': n,
        'quality_flags': flags or ['none'],
    }])


# ───────── Orchestrator ─────────
def compute_metric_scorecard(scenario_file: Path, metric_name: str = 'ttd') -> dict:
    """Runs §1 → §5 and returns four dataframes keyed by grain
    (observations → sub-fault → category → cumulative)."""
    observations = load_observations(scenario_file, metric_name)
    subfault = aggregate_subfault(observations)
    category = aggregate_category(observations)
    cumulative = aggregate_cumulative(observations, category)
    return {
        'observations': observations,
        'subfault': subfault,
        'category': category,
        'cumulative': cumulative,
    }


def show_pipeline(name: str, scenario_file: Path, metric_name: str = 'ttd', obs_head: int = 12):
    r = compute_metric_scorecard(scenario_file, metric_name)
    display(Markdown(f'### {name}  *(metric = `{metric_name}`)*'))

    display(Markdown(
        f'**§1+§2 — Observations (first {obs_head})**  \n'
        '_Classify each `(run, category, sub_fault)` as VALID / MISSING / INVALID_ZERO / NO_SLA, '
        'then normalize via piecewise curve: within SLA → `1 − 0.85·ratio`, breach → `max(0, 0.15 − 0.3·(ratio−1))`._'
    ))
    display(r['observations'].head(obs_head))

    display(Markdown(
        '**§3 — Sub-fault grain**  \n'
        '_Pool all obs per sub-fault (misses count as 0, NO_SLA excluded); confidence tier by n: '
        '<3 INSUFFICIENT, <5 LOW (mean), <20 MEDIUM (0.7·median + 0.3·p5), ≥20 HIGH (0.5·median + 0.3·p5 + 0.2·p1)._'
    ))
    display(r['subfault'])

    display(Markdown(
        '**§4 — Category grain**  \n'
        '_Pool all obs across sub-faults in the category; `category_score` = median of pooled scores '
        '(every obs equal weight so rare sub-faults aren\'t drowned by frequent ones)._'
    ))
    display(r['category'])

    display(Markdown(
        '**§5 — Cumulative (agent-level)**  \n'
        '_Headline `cumulative_score` = median of every pooled obs across all categories. '
        '`quality_flags` collapses all reliability diagnostics into one list: '
        '`skewed_distribution` (|mean − median| > 0.15), `mixed_category_health` (|mean(category_scores) − median| > 0.15), '
        '`N_obs_excluded_no_sla`. `[\'none\']` = trustworthy._'
    ))
    display(r['cumulative'])


sorted(p.name for p in SCENARIO_DIR.glob('scenario*.json'))


['scenario1_normal.json',
 'scenario2_null_heavy.json',
 'scenario3_low_variance.json',
 'scenario4_sla_breach.json',
 'scenario5_small_sample.json',
 'scenario6_category_imbalance.json',
 'scenario7a_small_n_baseline.json',
 'scenario7b_small_n_blind_median.json']

In [2]:
show_pipeline('Scenario 1 — NORMAL (all detects, well within SLA)',
              SCENARIO_DIR / 'scenario1_normal.json', metric_name='ttd')


### Scenario 1 — NORMAL (all detects, well within SLA)  *(metric = `ttd`)*

**§1+§2 — Observations (first 12)**  
_Classify each `(run, category, sub_fault)` as VALID / MISSING / INVALID_ZERO / NO_SLA, then normalize via piecewise curve: within SLA → `1 − 0.85·ratio`, breach → `max(0, 0.15 − 0.3·(ratio−1))`._

,run_id,category,sub_fault,metric_name,raw_value,sla,ratio,status,normalized_score,sla_compliant
0,936845b2,resource_fault,pod-cpu-hog,ttd,30.450,120,0.254,VALID,0.784,True
1,936845b2,resource_fault,pod-memory-hog,ttd,48.510,90,0.539,VALID,0.542,True
2,936845b2,network_fault,pod-network-loss,ttd,90.990,180,0.505,VALID,0.570,True
3,709c71d2,resource_fault,pod-cpu-hog,ttd,36.240,120,0.302,VALID,0.743,True
4,709c71d2,resource_fault,pod-memory-hog,ttd,35.840,90,0.398,VALID,0.662,True
5,709c71d2,network_fault,pod-network-loss,ttd,68.360,180,0.380,VALID,0.677,True
6,e1d41607,resource_fault,pod-cpu-hog,ttd,55.280,120,0.461,VALID,0.608,True
7,e1d41607,resource_fault,pod-memory-hog,ttd,46.390,90,0.515,VALID,0.562,True
8,e1d41607,network_fault,pod-network-loss,ttd,42.760,180,0.238,VALID,0.798,True
9,f59c7de2,resource_fault,pod-cpu-hog,ttd,25.360,120,0.211,VALID,0.820,True


**§3 — Sub-fault grain**  
_Pool all obs per sub-fault (misses count as 0, NO_SLA excluded); confidence tier by n: <3 INSUFFICIENT, <5 LOW (mean), <20 MEDIUM (0.7·median + 0.3·p5), ≥20 HIGH (0.5·median + 0.3·p5 + 0.2·p1)._

,category,sub_fault,n_attempted,detection_rate,sla_compliance,weighted_score,median,p5,p1,confidence
0,resource_fault,pod-cpu-hog,30,1.000,1.000,0.599,0.659,0.546,0.530,HIGH
1,resource_fault,pod-memory-hog,30,1.000,1.000,0.567,0.635,0.501,0.494,HIGH
2,network_fault,pod-network-loss,30,1.000,1.000,0.596,0.680,0.522,0.499,HIGH


**§4 — Category grain**  
_Pool all obs across sub-faults in the category; `category_score` = median of pooled scores (every obs equal weight so rare sub-faults aren't drowned by frequent ones)._

,category,n_sub_faults,n_attempted,detection_rate,sla_compliance,category_score
0,resource_fault,2,60,1.000,1.000,0.656
1,network_fault,1,30,1.000,1.000,0.680


**§5 — Cumulative (agent-level)**  
_Headline `cumulative_score` = median of every pooled obs across all categories. `quality_flags` collapses all reliability diagnostics into one list: `skewed_distribution` (|mean − median| > 0.15), `mixed_category_health` (|mean(category_scores) − median| > 0.15), `N_obs_excluded_no_sla`. `['none']` = trustworthy._

,cumulative_score,detection_rate,sla_compliance,n_attempted,quality_flags
0,0.660,1.000,1.000,90,[none]


In [3]:
show_pipeline('Scenario 2 — NULL-HEAVY (~70% MISSING)',
              SCENARIO_DIR / 'scenario2_null_heavy.json', metric_name='ttd')


### Scenario 2 — NULL-HEAVY (~70% MISSING)  *(metric = `ttd`)*

**§1+§2 — Observations (first 12)**  
_Classify each `(run, category, sub_fault)` as VALID / MISSING / INVALID_ZERO / NO_SLA, then normalize via piecewise curve: within SLA → `1 − 0.85·ratio`, breach → `max(0, 0.15 − 0.3·(ratio−1))`._

,run_id,category,sub_fault,metric_name,raw_value,sla,ratio,status,normalized_score,sla_compliant
0,829912c9,resource_fault,pod-cpu-hog,ttd,92.240,120,0.769,VALID,0.347,True
1,829912c9,resource_fault,pod-memory-hog,ttd,NaN,90,NaN,MISSING,0.000,False
2,829912c9,network_fault,pod-network-loss,ttd,NaN,180,NaN,MISSING,0.000,False
3,eadd1cf1,resource_fault,pod-cpu-hog,ttd,76.990,120,0.642,VALID,0.455,True
4,eadd1cf1,resource_fault,pod-memory-hog,ttd,NaN,90,NaN,MISSING,0.000,False
5,eadd1cf1,network_fault,pod-network-loss,ttd,NaN,180,NaN,MISSING,0.000,False
6,c4114481,resource_fault,pod-cpu-hog,ttd,NaN,120,NaN,MISSING,0.000,False
7,c4114481,resource_fault,pod-memory-hog,ttd,NaN,90,NaN,MISSING,0.000,False
8,c4114481,network_fault,pod-network-loss,ttd,NaN,180,NaN,MISSING,0.000,False
9,f02a9098,resource_fault,pod-cpu-hog,ttd,NaN,120,NaN,MISSING,0.000,False


**§3 — Sub-fault grain**  
_Pool all obs per sub-fault (misses count as 0, NO_SLA excluded); confidence tier by n: <3 INSUFFICIENT, <5 LOW (mean), <20 MEDIUM (0.7·median + 0.3·p5), ≥20 HIGH (0.5·median + 0.3·p5 + 0.2·p1)._

,category,sub_fault,n_attempted,detection_rate,sla_compliance,weighted_score,median,p5,p1,confidence
0,resource_fault,pod-cpu-hog,30,0.400,0.400,0.000,0.000,0.000,0.000,HIGH
1,resource_fault,pod-memory-hog,30,0.367,0.367,0.000,0.000,0.000,0.000,HIGH
2,network_fault,pod-network-loss,30,0.233,0.233,0.000,0.000,0.000,0.000,HIGH


**§4 — Category grain**  
_Pool all obs across sub-faults in the category; `category_score` = median of pooled scores (every obs equal weight so rare sub-faults aren't drowned by frequent ones)._

,category,n_sub_faults,n_attempted,detection_rate,sla_compliance,category_score
0,resource_fault,2,60,0.383,0.383,0.000
1,network_fault,1,30,0.233,0.233,0.000


**§5 — Cumulative (agent-level)**  
_Headline `cumulative_score` = median of every pooled obs across all categories. `quality_flags` collapses all reliability diagnostics into one list: `skewed_distribution` (|mean − median| > 0.15), `mixed_category_health` (|mean(category_scores) − median| > 0.15), `N_obs_excluded_no_sla`. `['none']` = trustworthy._

,cumulative_score,detection_rate,sla_compliance,n_attempted,quality_flags
0,0.000,0.333,0.333,90,[skewed_distribution]


In [4]:
show_pipeline('Scenario 3 — LOW VARIANCE (tight cluster ~40% of SLA)',
              SCENARIO_DIR / 'scenario3_low_variance.json', metric_name='ttd')


### Scenario 3 — LOW VARIANCE (tight cluster ~40% of SLA)  *(metric = `ttd`)*

**§1+§2 — Observations (first 12)**  
_Classify each `(run, category, sub_fault)` as VALID / MISSING / INVALID_ZERO / NO_SLA, then normalize via piecewise curve: within SLA → `1 − 0.85·ratio`, breach → `max(0, 0.15 − 0.3·(ratio−1))`._

,run_id,category,sub_fault,metric_name,raw_value,sla,ratio,status,normalized_score,sla_compliant
0,595dd9d9,resource_fault,pod-cpu-hog,ttd,48.340,120,0.403,VALID,0.658,True
1,595dd9d9,resource_fault,pod-memory-hog,ttd,39.380,90,0.438,VALID,0.628,True
2,595dd9d9,network_fault,pod-network-loss,ttd,66.970,180,0.372,VALID,0.684,True
3,dc9f49d9,resource_fault,pod-cpu-hog,ttd,51.570,120,0.430,VALID,0.635,True
4,dc9f49d9,resource_fault,pod-memory-hog,ttd,35.300,90,0.392,VALID,0.667,True
5,dc9f49d9,network_fault,pod-network-loss,ttd,70.590,180,0.392,VALID,0.667,True
6,7c3b96cd,resource_fault,pod-cpu-hog,ttd,54.840,120,0.457,VALID,0.612,True
7,7c3b96cd,resource_fault,pod-memory-hog,ttd,36.430,90,0.405,VALID,0.656,True
8,7c3b96cd,network_fault,pod-network-loss,ttd,71.770,180,0.399,VALID,0.661,True
9,4a97ffa2,resource_fault,pod-cpu-hog,ttd,50.630,120,0.422,VALID,0.641,True


**§3 — Sub-fault grain**  
_Pool all obs per sub-fault (misses count as 0, NO_SLA excluded); confidence tier by n: <3 INSUFFICIENT, <5 LOW (mean), <20 MEDIUM (0.7·median + 0.3·p5), ≥20 HIGH (0.5·median + 0.3·p5 + 0.2·p1)._

,category,sub_fault,n_attempted,detection_rate,sla_compliance,weighted_score,median,p5,p1,confidence
0,resource_fault,pod-cpu-hog,30,1.000,1.000,0.641,0.664,0.622,0.612,HIGH
1,resource_fault,pod-memory-hog,30,1.000,1.000,0.643,0.667,0.623,0.613,HIGH
2,network_fault,pod-network-loss,30,1.000,1.000,0.640,0.664,0.619,0.610,HIGH


**§4 — Category grain**  
_Pool all obs across sub-faults in the category; `category_score` = median of pooled scores (every obs equal weight so rare sub-faults aren't drowned by frequent ones)._

,category,n_sub_faults,n_attempted,detection_rate,sla_compliance,category_score
0,resource_fault,2,60,1.000,1.000,0.666
1,network_fault,1,30,1.000,1.000,0.664


**§5 — Cumulative (agent-level)**  
_Headline `cumulative_score` = median of every pooled obs across all categories. `quality_flags` collapses all reliability diagnostics into one list: `skewed_distribution` (|mean − median| > 0.15), `mixed_category_health` (|mean(category_scores) − median| > 0.15), `N_obs_excluded_no_sla`. `['none']` = trustworthy._

,cumulative_score,detection_rate,sla_compliance,n_attempted,quality_flags
0,0.665,1.000,1.000,90,[none]


In [5]:
show_pipeline('Scenario 4 — SLA BREACH (ratios > 1.0, some SEVERE > 1.5)',
              SCENARIO_DIR / 'scenario4_sla_breach.json', metric_name='ttd')


### Scenario 4 — SLA BREACH (ratios > 1.0, some SEVERE > 1.5)  *(metric = `ttd`)*

**§1+§2 — Observations (first 12)**  
_Classify each `(run, category, sub_fault)` as VALID / MISSING / INVALID_ZERO / NO_SLA, then normalize via piecewise curve: within SLA → `1 − 0.85·ratio`, breach → `max(0, 0.15 − 0.3·(ratio−1))`._

,run_id,category,sub_fault,metric_name,raw_value,sla,ratio,status,normalized_score,sla_compliant
0,59834e01,resource_fault,pod-cpu-hog,ttd,66.190,120,0.552,VALID,0.531,True
1,59834e01,resource_fault,pod-memory-hog,ttd,94.180,90,1.046,VALID,0.136,False
2,59834e01,network_fault,pod-network-loss,ttd,NaN,180,NaN,MISSING,0.000,False
3,ce66247e,resource_fault,pod-cpu-hog,ttd,153.050,120,1.275,VALID,0.067,False
4,ce66247e,resource_fault,pod-memory-hog,ttd,238.300,90,2.648,VALID,0.000,False
5,ce66247e,network_fault,pod-network-loss,ttd,138.300,180,0.768,VALID,0.347,True
6,85312f0d,resource_fault,pod-cpu-hog,ttd,70.360,120,0.586,VALID,0.502,True
7,85312f0d,resource_fault,pod-memory-hog,ttd,54.650,90,0.607,VALID,0.484,True
8,85312f0d,network_fault,pod-network-loss,ttd,493.810,180,2.743,VALID,0.000,False
9,08d6aa9d,resource_fault,pod-cpu-hog,ttd,324.080,120,2.701,VALID,0.000,False


**§3 — Sub-fault grain**  
_Pool all obs per sub-fault (misses count as 0, NO_SLA excluded); confidence tier by n: <3 INSUFFICIENT, <5 LOW (mean), <20 MEDIUM (0.7·median + 0.3·p5), ≥20 HIGH (0.5·median + 0.3·p5 + 0.2·p1)._

,category,sub_fault,n_attempted,detection_rate,sla_compliance,weighted_score,median,p5,p1,confidence
0,resource_fault,pod-cpu-hog,30,1.000,0.233,0.017,0.034,0.000,0.000,HIGH
1,resource_fault,pod-memory-hog,30,0.900,0.233,0.051,0.101,0.000,0.000,HIGH
2,network_fault,pod-network-loss,30,0.800,0.267,0.015,0.031,0.000,0.000,HIGH


**§4 — Category grain**  
_Pool all obs across sub-faults in the category; `category_score` = median of pooled scores (every obs equal weight so rare sub-faults aren't drowned by frequent ones)._

,category,n_sub_faults,n_attempted,detection_rate,sla_compliance,category_score
0,resource_fault,2,60,0.950,0.233,0.074
1,network_fault,1,30,0.800,0.267,0.031


**§5 — Cumulative (agent-level)**  
_Headline `cumulative_score` = median of every pooled obs across all categories. `quality_flags` collapses all reliability diagnostics into one list: `skewed_distribution` (|mean − median| > 0.15), `mixed_category_health` (|mean(category_scores) − median| > 0.15), `N_obs_excluded_no_sla`. `['none']` = trustworthy._

,cumulative_score,detection_rate,sla_compliance,n_attempted,quality_flags
0,0.068,0.900,0.244,90,[none]


In [6]:
show_pipeline('Scenario 5 — SMALL SAMPLE (5 runs)',
              SCENARIO_DIR / 'scenario5_small_sample.json', metric_name='ttd', obs_head=15)


### Scenario 5 — SMALL SAMPLE (5 runs)  *(metric = `ttd`)*

**§1+§2 — Observations (first 15)**  
_Classify each `(run, category, sub_fault)` as VALID / MISSING / INVALID_ZERO / NO_SLA, then normalize via piecewise curve: within SLA → `1 − 0.85·ratio`, breach → `max(0, 0.15 − 0.3·(ratio−1))`._

,run_id,category,sub_fault,metric_name,raw_value,sla,ratio,status,normalized_score,sla_compliant
0,0675e991,resource_fault,pod-cpu-hog,ttd,66.960,120,0.558,VALID,0.526,True
1,0675e991,resource_fault,pod-memory-hog,ttd,60.150,90,0.668,VALID,0.432,True
2,0675e991,network_fault,pod-network-loss,ttd,118.310,180,0.657,VALID,0.441,True
3,ec96e7f9,resource_fault,pod-cpu-hog,ttd,NaN,120,NaN,MISSING,0.000,False
4,ec96e7f9,resource_fault,pod-memory-hog,ttd,60.200,90,0.669,VALID,0.431,True
5,ec96e7f9,network_fault,pod-network-loss,ttd,116.190,180,0.645,VALID,0.451,True
6,424fb8cc,resource_fault,pod-cpu-hog,ttd,NaN,120,NaN,MISSING,0.000,False
7,424fb8cc,resource_fault,pod-memory-hog,ttd,25.710,90,0.286,VALID,0.757,True
8,424fb8cc,network_fault,pod-network-loss,ttd,83.820,180,0.466,VALID,0.604,True
9,dcad3a54,resource_fault,pod-cpu-hog,ttd,NaN,120,NaN,MISSING,0.000,False


**§3 — Sub-fault grain**  
_Pool all obs per sub-fault (misses count as 0, NO_SLA excluded); confidence tier by n: <3 INSUFFICIENT, <5 LOW (mean), <20 MEDIUM (0.7·median + 0.3·p5), ≥20 HIGH (0.5·median + 0.3·p5 + 0.2·p1)._

,category,sub_fault,n_attempted,detection_rate,sla_compliance,weighted_score,median,p5,p1,confidence
0,resource_fault,pod-cpu-hog,5,0.400,0.400,0.000,0.000,0.000,0.000,MEDIUM
1,resource_fault,pod-memory-hog,5,0.800,0.800,0.328,0.432,0.086,0.017,MEDIUM
2,network_fault,pod-network-loss,5,1.000,1.000,0.493,0.515,0.443,0.442,MEDIUM


**§4 — Category grain**  
_Pool all obs across sub-faults in the category; `category_score` = median of pooled scores (every obs equal weight so rare sub-faults aren't drowned by frequent ones)._

,category,n_sub_faults,n_attempted,detection_rate,sla_compliance,category_score
0,resource_fault,2,10,0.600,0.600,0.432
1,network_fault,1,5,1.000,1.000,0.515


**§5 — Cumulative (agent-level)**  
_Headline `cumulative_score` = median of every pooled obs across all categories. `quality_flags` collapses all reliability diagnostics into one list: `skewed_distribution` (|mean − median| > 0.15), `mixed_category_health` (|mean(category_scores) − median| > 0.15), `N_obs_excluded_no_sla`. `['none']` = trustworthy._

,cumulative_score,detection_rate,sla_compliance,n_attempted,quality_flags
0,0.451,0.733,0.733,15,[none]


In [7]:
show_pipeline('Scenario 6 — CATEGORY IMBALANCE (network=null-heavy + resource=low-variance)',
              SCENARIO_DIR / 'scenario6_category_imbalance.json', metric_name='ttd')

### Scenario 6 — CATEGORY IMBALANCE (network=null-heavy + resource=low-variance)  *(metric = `ttd`)*

**§1+§2 — Observations (first 12)**  
_Classify each `(run, category, sub_fault)` as VALID / MISSING / INVALID_ZERO / NO_SLA, then normalize via piecewise curve: within SLA → `1 − 0.85·ratio`, breach → `max(0, 0.15 − 0.3·(ratio−1))`._

,run_id,category,sub_fault,metric_name,raw_value,sla,ratio,status,normalized_score,sla_compliant
0,67da67f6,resource_fault,pod-cpu-hog,ttd,55.810,120,0.465,VALID,0.605,True
1,67da67f6,resource_fault,pod-memory-hog,ttd,33.010,90,0.367,VALID,0.688,True
2,67da67f6,network_fault,pod-network-loss,ttd,45.020,180,0.250,VALID,0.787,True
3,85034030,resource_fault,pod-cpu-hog,ttd,44.830,120,0.374,VALID,0.682,True
4,85034030,resource_fault,pod-memory-hog,ttd,34.450,90,0.383,VALID,0.675,True
5,85034030,network_fault,pod-network-loss,ttd,NaN,180,NaN,MISSING,0.000,False
6,be60bcad,resource_fault,pod-cpu-hog,ttd,49.610,120,0.413,VALID,0.649,True
7,be60bcad,resource_fault,pod-memory-hog,ttd,37.290,90,0.414,VALID,0.648,True
8,be60bcad,network_fault,pod-network-loss,ttd,NaN,180,NaN,MISSING,0.000,False
9,e3fb26f9,resource_fault,pod-cpu-hog,ttd,48.720,120,0.406,VALID,0.655,True


**§3 — Sub-fault grain**  
_Pool all obs per sub-fault (misses count as 0, NO_SLA excluded); confidence tier by n: <3 INSUFFICIENT, <5 LOW (mean), <20 MEDIUM (0.7·median + 0.3·p5), ≥20 HIGH (0.5·median + 0.3·p5 + 0.2·p1)._

,category,sub_fault,n_attempted,detection_rate,sla_compliance,weighted_score,median,p5,p1,confidence
0,resource_fault,pod-cpu-hog,30,1.000,1.000,0.631,0.655,0.609,0.603,HIGH
1,resource_fault,pod-memory-hog,30,1.000,1.000,0.638,0.653,0.628,0.619,HIGH
2,network_fault,pod-network-loss,30,0.333,0.333,0.000,0.000,0.000,0.000,HIGH


**§4 — Category grain**  
_Pool all obs across sub-faults in the category; `category_score` = median of pooled scores (every obs equal weight so rare sub-faults aren't drowned by frequent ones)._

,category,n_sub_faults,n_attempted,detection_rate,sla_compliance,category_score
0,resource_fault,2,60,1.000,1.000,0.654
1,network_fault,1,30,0.333,0.333,0.000


**§5 — Cumulative (agent-level)**  
_Headline `cumulative_score` = median of every pooled obs across all categories. `quality_flags` collapses all reliability diagnostics into one list: `skewed_distribution` (|mean − median| > 0.15), `mixed_category_health` (|mean(category_scores) − median| > 0.15), `N_obs_excluded_no_sla`. `['none']` = trustworthy._

,cumulative_score,detection_rate,sla_compliance,n_attempted,quality_flags
0,0.644,0.778,0.778,90,[mixed_category_health]


In [8]:
show_pipeline('Scenario 7a — SMALL N BASELINE (n=3, all good: TTD = 0.3/0.4/0.5 x SLA)',
              SCENARIO_DIR / 'scenario7a_small_n_baseline.json', metric_name='ttd')

### Scenario 7a — SMALL N BASELINE (n=3, all good: TTD = 0.3/0.4/0.5 x SLA)  *(metric = `ttd`)*

**§1+§2 — Observations (first 12)**  
_Classify each `(run, category, sub_fault)` as VALID / MISSING / INVALID_ZERO / NO_SLA, then normalize via piecewise curve: within SLA → `1 − 0.85·ratio`, breach → `max(0, 0.15 − 0.3·(ratio−1))`._

,run_id,category,sub_fault,metric_name,raw_value,sla,ratio,status,normalized_score,sla_compliant
0,run-1,network_fault,pod-network-loss,ttd,54.000,180.000,0.300,VALID,0.745,True
1,run-2,network_fault,pod-network-loss,ttd,72.000,180.000,0.400,VALID,0.660,True
2,run-3,network_fault,pod-network-loss,ttd,90.000,180.000,0.500,VALID,0.575,True


**§3 — Sub-fault grain**  
_Pool all obs per sub-fault (misses count as 0, NO_SLA excluded); confidence tier by n: <3 INSUFFICIENT, <5 LOW (mean), <20 MEDIUM (0.7·median + 0.3·p5), ≥20 HIGH (0.5·median + 0.3·p5 + 0.2·p1)._

,category,sub_fault,n_attempted,detection_rate,sla_compliance,weighted_score,median,p5,p1,confidence
0,network_fault,pod-network-loss,3,1.000,1.000,0.660,0.660,0.583,0.577,LOW


**§4 — Category grain**  
_Pool all obs across sub-faults in the category; `category_score` = median of pooled scores (every obs equal weight so rare sub-faults aren't drowned by frequent ones)._

,category,n_sub_faults,n_attempted,detection_rate,sla_compliance,category_score
0,network_fault,1,3,1.000,1.000,0.660


**§5 — Cumulative (agent-level)**  
_Headline `cumulative_score` = median of every pooled obs across all categories. `quality_flags` collapses all reliability diagnostics into one list: `skewed_distribution` (|mean − median| > 0.15), `mixed_category_health` (|mean(category_scores) − median| > 0.15), `N_obs_excluded_no_sla`. `['none']` = trustworthy._

,cumulative_score,detection_rate,sla_compliance,n_attempted,quality_flags
0,0.660,1.000,1.000,3,[none]


In [9]:
show_pipeline('Scenario 7b — SMALL N BLIND MEDIAN (n=3, last obs breaches: TTD = 0.3/0.4/1.5 x SLA)',
              SCENARIO_DIR / 'scenario7b_small_n_blind_median.json', metric_name='ttd')

### Scenario 7b — SMALL N BLIND MEDIAN (n=3, last obs breaches: TTD = 0.3/0.4/1.5 x SLA)  *(metric = `ttd`)*

**§1+§2 — Observations (first 12)**  
_Classify each `(run, category, sub_fault)` as VALID / MISSING / INVALID_ZERO / NO_SLA, then normalize via piecewise curve: within SLA → `1 − 0.85·ratio`, breach → `max(0, 0.15 − 0.3·(ratio−1))`._

,run_id,category,sub_fault,metric_name,raw_value,sla,ratio,status,normalized_score,sla_compliant
0,run-1,network_fault,pod-network-loss,ttd,54.000,180.000,0.300,VALID,0.745,True
1,run-2,network_fault,pod-network-loss,ttd,72.000,180.000,0.400,VALID,0.660,True
2,run-3,network_fault,pod-network-loss,ttd,270.000,180.000,1.500,VALID,0.000,False


**§3 — Sub-fault grain**  
_Pool all obs per sub-fault (misses count as 0, NO_SLA excluded); confidence tier by n: <3 INSUFFICIENT, <5 LOW (mean), <20 MEDIUM (0.7·median + 0.3·p5), ≥20 HIGH (0.5·median + 0.3·p5 + 0.2·p1)._

,category,sub_fault,n_attempted,detection_rate,sla_compliance,weighted_score,median,p5,p1,confidence
0,network_fault,pod-network-loss,3,1.000,0.667,0.468,0.660,0.066,0.013,LOW


**§4 — Category grain**  
_Pool all obs across sub-faults in the category; `category_score` = median of pooled scores (every obs equal weight so rare sub-faults aren't drowned by frequent ones)._

,category,n_sub_faults,n_attempted,detection_rate,sla_compliance,category_score
0,network_fault,1,3,1.000,0.667,0.660


**§5 — Cumulative (agent-level)**  
_Headline `cumulative_score` = median of every pooled obs across all categories. `quality_flags` collapses all reliability diagnostics into one list: `skewed_distribution` (|mean − median| > 0.15), `mixed_category_health` (|mean(category_scores) − median| > 0.15), `N_obs_excluded_no_sla`. `['none']` = trustworthy._

,cumulative_score,detection_rate,sla_compliance,n_attempted,quality_flags
0,0.660,1.000,0.667,3,[skewed_distribution]


In [10]:
# Real TTM data — same pipeline, just swap metric_name and the input JSON.
TTM_FILE = Path('./data/ttm_by_run_category_fault.json')
show_pipeline('Real data — TTM (12-05-26-sequential-aarya-30run)',
              TTM_FILE, metric_name='ttm')

C:\Users\meemankgupta\AppData\Local\Temp\ipykernel_43576\3662417325.py:83: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  n_compliant = int(g['sla_compliant'].fillna(False).sum())
C:\Users\meemankgupta\AppData\Local\Temp\ipykernel_43576\3662417325.py:119: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  n_compliant = int(g['sla_compliant'].fillna(False).sum())
C:\Users\meemankgupta\AppData\Local\Temp\ipykernel_43576\3662417325.py:139: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. 

### Real data — TTM (12-05-26-sequential-aarya-30run)  *(metric = `ttm`)*

**§1+§2 — Observations (first 12)**  
_Classify each `(run, category, sub_fault)` as VALID / MISSING / INVALID_ZERO / NO_SLA, then normalize via piecewise curve: within SLA → `1 − 0.85·ratio`, breach → `max(0, 0.15 − 0.3·(ratio−1))`._

,run_id,category,sub_fault,metric_name,raw_value,sla,ratio,status,normalized_score,sla_compliant
0,000098ae,resource_fault,pod-cpu-hog,ttm,None,300.000,None,MISSING,0.000,False
1,000098ae,resource_fault,pod-memory-hog,ttm,None,240.000,None,MISSING,0.000,False
2,000098ae,network_fault,pod-network-loss,ttm,None,360.000,None,MISSING,0.000,False
3,0a0e9fcf,resource_fault,pod-cpu-hog,ttm,None,300.000,None,MISSING,0.000,False
4,0a0e9fcf,resource_fault,pod-memory-hog,ttm,None,240.000,None,MISSING,0.000,False
5,0a0e9fcf,network_fault,pod-network-loss,ttm,None,360.000,None,MISSING,0.000,False
6,0ac7ac9c,resource_fault,pod-cpu-hog,ttm,None,300.000,None,MISSING,0.000,False
7,0ac7ac9c,resource_fault,pod-memory-hog,ttm,None,240.000,None,MISSING,0.000,False
8,0ac7ac9c,network_fault,pod-network-loss,ttm,None,360.000,None,MISSING,0.000,False
9,0dd90da1,resource_fault,pod-cpu-hog,ttm,None,300.000,None,MISSING,0.000,False


**§3 — Sub-fault grain**  
_Pool all obs per sub-fault (misses count as 0, NO_SLA excluded); confidence tier by n: <3 INSUFFICIENT, <5 LOW (mean), <20 MEDIUM (0.7·median + 0.3·p5), ≥20 HIGH (0.5·median + 0.3·p5 + 0.2·p1)._

,category,sub_fault,n_attempted,detection_rate,sla_compliance,weighted_score,median,p5,p1,confidence
0,resource_fault,pod-cpu-hog,31,0.000,0.000,0.000,0.000,0.000,0.000,HIGH
1,resource_fault,pod-memory-hog,31,0.000,0.000,0.000,0.000,0.000,0.000,HIGH
2,network_fault,pod-network-loss,31,0.000,0.000,0.000,0.000,0.000,0.000,HIGH


**§4 — Category grain**  
_Pool all obs across sub-faults in the category; `category_score` = median of pooled scores (every obs equal weight so rare sub-faults aren't drowned by frequent ones)._

,category,n_sub_faults,n_attempted,detection_rate,sla_compliance,category_score
0,resource_fault,2,62,0.000,0.000,0.000
1,network_fault,1,31,0.000,0.000,0.000


**§5 — Cumulative (agent-level)**  
_Headline `cumulative_score` = median of every pooled obs across all categories. `quality_flags` collapses all reliability diagnostics into one list: `skewed_distribution` (|mean − median| > 0.15), `mixed_category_health` (|mean(category_scores) − median| > 0.15), `N_obs_excluded_no_sla`. `['none']` = trustworthy._

,cumulative_score,detection_rate,sla_compliance,n_attempted,quality_flags
0,0.000,0.000,0.000,93,[9_obs_excluded_no_sla]
